# 10 -- Baseline vs. candidate release comparison

Paired script: `analysis/compare_releases.py`.

**Corrected, 2026-07-22 Codex review finding (fifth round, R5F17): this notebook previously
claimed BOTH the win-rate and R-expectancy differences use a two-sample bootstrap interval.
That was only ever true for R-expectancy.** The win-rate difference uses a proper
two-proportion interval (Newcombe-Wilson, `metrics.wilson_diff_confidence_interval`) --
bootstrapping raw 0/1 outcomes collapses to a degenerate interval at boundary samples (e.g.
all-win vs. all-loss groups), which is not a defensible uncertainty statement for a
proportion (see `compare_releases.py`'s own module docstring and the comment above its
`win_rate_diff` computation). Only the continuous R-expectancy difference remains a genuine
two-sample bootstrap (`two_sample_bootstrap_diff`), appropriate for a continuous statistic.
The cell below prints and asserts `summary["win_rate_diff"]["method"]` so this claim is
independently checked against the actual composed output, not just described in prose.

Per the reproducibility contract's "tiny samples cannot drive automatic changes" rule, this
never declares a release "better" automatically -- it reports a difference and its CI; the
go/no-go judgment remains a human decision.

**Uses clearly-labelled SYNTHETIC trade data for both releases.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.compare_releases import run

In [ ]:
def make_trades(path, exits, profits):
    rows = [
        {
            "trade_id": f"t{i}",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T00:00:00Z",
            "exit_time": "2026-07-21T01:00:00Z",
            "entry_price": 100.0,
            "exit_price": e,
            "stop_price": 98.0,
            "profit": p,
        }
        for i, (e, p) in enumerate(zip(exits, profits))
    ]
    pd.DataFrame(rows).to_csv(path, index=False)


tmp_dir = Path(tempfile.mkdtemp(prefix="themba_compare_demo_"))
# Baseline: 25% win rate (5 wins / 15 losses out of 20). Candidate: 75% (15/20).
make_trades(tmp_dir / "baseline.csv", [95.0] * 15 + [105.0] * 5, [-10.0] * 15 + [10.0] * 5)
make_trades(tmp_dir / "candidate.csv", [105.0] * 15 + [95.0] * 5, [10.0] * 15 + [-10.0] * 5)

In [ ]:
summary = run(
    tmp_dir / "baseline.csv",
    tmp_dir / "candidate.csv",
    n_resamples=2000,
    seed=1,
    # period_start/period_end are now REQUIRED (Codex review finding,
    # 2026-07-22, fourth round): every trade in both datasets must fall
    # inside this caller-claimed comparison window.
    period_start="2026-01-01T00:00:00Z",
    period_end="2026-12-31T23:59:59Z",
    # broker/timeframe/modelling_mode/set_file/market_data_id/spread_note/
    # slippage_note are now REQUIRED and cross-checked for equality
    # (Codex review finding, 2026-07-22, fifth round) -- a complete
    # comparability manifest, not an optional best-effort assertion.
    baseline_broker="Deriv",
    candidate_broker="Deriv",
    baseline_timeframe="M5",
    candidate_timeframe="M5",
    baseline_modelling_mode="every_tick",
    candidate_modelling_mode="every_tick",
    baseline_set_file="default.set",
    candidate_set_file="default.set",
    baseline_market_data_id="synthetic-fixture-v1",
    candidate_market_data_id="synthetic-fixture-v1",
    baseline_spread_note="2-pip fixed spread assumed",
    candidate_spread_note="2-pip fixed spread assumed",
    baseline_slippage_note="no slippage modelled",
    candidate_slippage_note="no slippage modelled",
    output_json=tmp_dir / "compare.json",
    repo_path=PROJECT_ROOT.parents[1],
)

print(f"baseline_win_rate    = {summary['baseline_win_rate']:.4f}")
print(f"candidate_win_rate   = {summary['candidate_win_rate']:.4f}")
print(
    f"win_rate_diff        = {summary['win_rate_diff']['observed_diff']:.4f} "
    f"(95% CI [{summary['win_rate_diff']['ci_lower']:.4f}, {summary['win_rate_diff']['ci_upper']:.4f}])"
)
print(f"likely_significant   = {summary['win_rate_diff']['likely_significant']}")

assert abs(summary["baseline_win_rate"] - 0.25) < 1e-9
assert abs(summary["candidate_win_rate"] - 0.75) < 1e-9
assert summary["win_rate_diff"]["likely_significant"] is True

# **Added, 2026-07-22 Codex review finding (fifth round, R5F17): hand-check
# the ACTUAL mechanism each difference uses, not just that the cell ran.**
print(f"win_rate_diff method = {summary['win_rate_diff']['method']!r}")
assert summary["win_rate_diff"]["method"] == "newcombe_wilson"

# R-expectancy diff: every baseline trade has entry=100, stop=98, so
# r_multiple = (exit_price - 100) / (100 - 98) = (exit_price - 100) / 2.
# Baseline: 15 trades exit=95.0 -> r=-2.5, 5 trades exit=105.0 -> r=+2.5;
# mean = (15*(-2.5) + 5*(2.5)) / 20 = -1.25.
# Candidate: 15 trades exit=105.0 -> r=+2.5, 5 trades exit=95.0 -> r=-2.5;
# mean = (15*(2.5) + 5*(-2.5)) / 20 = +1.25.
# observed_diff = mean(candidate) - mean(baseline) = 1.25 - (-1.25) = 2.5 --
# an exact point statistic on the real data (not a resampled quantity, see
# two_sample_bootstrap_diff's own docstring), so this is asserted exactly;
# the resampled CI bounds are stochastic (though seeded) and are printed,
# not hard-asserted, but likely_significant is still hand-derivable: +-2.5
# is a total separation between the two groups, so no resample of either
# group (each drawn only from {-2.5, +2.5} or {+2.5, -2.5}) can plausibly
# produce a CI straddling 0.
print(
    f"expectancy_r_diff         = {summary['expectancy_r_diff']['observed_diff']:.4f} "
    f"(95% CI [{summary['expectancy_r_diff']['ci_lower']:.4f}, "
    f"{summary['expectancy_r_diff']['ci_upper']:.4f}])"
)
assert abs(summary["baseline_expectancy_r"] - (-1.25)) < 1e-9
assert abs(summary["candidate_expectancy_r"] - 1.25) < 1e-9
assert abs(summary["expectancy_r_diff"]["observed_diff"] - 2.5) < 1e-9
assert summary["expectancy_r_diff"]["likely_significant"] is True

## Real-data run: PENDING

Requires two real trade histories (baseline release vs. a candidate release) -- neither exists yet.